In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import torch.nn as nn
import numpy as np
from sklearn.metrics import r2_score, root_mean_squared_error
import pandas as pd
import pickle
import sys

In [ ]:
class AttentionFusionModel(nn.Module):
    def __init__(self, gene_dim, chem_dim, hidden_dim=256, num_heads=4, dropout=0.2):
        """
        Args:
            gene_dim (int): Number of features in the gene expression data (e.g., 978 for L1000).
            chem_dim (int): Number of features in the chemical descriptors (e.g., 2048 for Morgan + physchem).
            hidden_dim (int): Size of the shared hidden dimension for attention.
            num_heads (int): Number of attention heads.
            dropout (float): Dropout rate to prevent overfitting.
        """
        super().__init__()
        
        # projection layers to map gene and chem features in the same hidden space
        self.gene_proj = nn.Linear(gene_dim, hidden_dim)
        self.chem_proj = nn.Linear(chem_dim, hidden_dim)
        
        # Cross-attention layers
        # Genomic features conditionally on chemical input
        self.cross_attn_drug_gene = nn.MultiheadAttention(
            embed_dim=hidden_dim, num_heads=num_heads, dropout=dropout, batch_first=True
        )
        
        # Chemical features conditionally on genomic input
        self.cross_attn_gene_drug = nn.MultiheadAttention(
            embed_dim=hidden_dim, num_heads=num_heads, dropout=dropout, batch_first=True
        )
        
        # forward network (MLP) for final prediction (z.B. IC50, AUC)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, gene_features, chem_features):
        # Projection: (Batch, Feature_Dim) -> (Batch, Hidden_Dim)
        h_gene = self.gene_proj(gene_features)
        h_chem = self.chem_proj(chem_features)
        
        # PyTorch MultiheadAttention expects sequences. 
        # add sequence dimension: (Batch, Hidden_Dim) -> (Batch, 1, Hidden_Dim)
        h_gene_seq = h_gene.unsqueeze(1)
        h_chem_seq = h_chem.unsqueeze(1)
        
        # compute cross-attention
        # Query: Chem, Key/Value: Gene
        attn_chem, _ = self.cross_attn_drug_gene(query=h_chem_seq, key=h_gene_seq, value=h_gene_seq)
        
        # Query: Gene, Key/Value: Chem
        attn_gene, _ = self.cross_attn_gene_drug(query=h_gene_seq, key=h_chem_seq, value=h_chem_seq)
        
        # remove sequence dimension: (Batch, 1, Hidden_Dim) -> (Batch, Hidden_Dim)
        attn_chem = attn_chem.squeeze(1)
        attn_gene = attn_gene.squeeze(1)
        
        # concatenate attention-based representations
        fused_features = torch.cat([attn_gene, attn_chem], dim=1) # Form: (Batch, Hidden_Dim * 2)
        
        # final prediction by MLP
        prediction = self.fc(fused_features)
        
        return prediction.squeeze(-1)

Der DataLoader kümmert sich darum, dass das Modell die Daten in Batches bekommt

In [ ]:
!{sys.executable} -m pip install -q "pyarrow>=13.0.0"

df = pd.read_pickle(
    r"C:\Users\Juli\Documents\Master\Projekt Genomforschung\Datasets\harmonized_data.pkl"
)

In [12]:
# training data
from sklearn.model_selection import train_test_split

target = 'LN_IC50'
pharmacophores = ['Donor', 'Acceptor', 'Aromatic', 'Hydrophobe', 'LumpedHydrophobe', 'PosIonizable', 'NegIonizable', 'ZnBinder']
X_genomic = df.filter(regex=r'.* \(.*\)').values.astype('float32')
X_chem = df[pharmacophores + list(df.columns[df.columns.str.startswith('Bit_')])].values.astype('float32')
y = df[target].values.astype('float32')

idx_train, idx_test = train_test_split(range(len(y)), test_size=0.2, random_state=42)

In [13]:
class DrugResponseDataset(Dataset):
    def __init__(self, X_gen, X_ch, labels, indices):
        self.X_genomic = torch.tensor(X_gen[indices], dtype=torch.float32)
        self.X_chem = torch.tensor(X_ch[indices], dtype=torch.float32)
        self.y = torch.tensor(labels[indices], dtype=torch.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.X_genomic[idx], self.X_chem[idx], self.y[idx]

train_loader = DataLoader(DrugResponseDataset(X_genomic, X_chem, y, idx_train), batch_size=32, shuffle=True)
test_loader = DataLoader(DrugResponseDataset(X_genomic, X_chem, y, idx_test), batch_size=64, shuffle=False)

In [ ]:
# initialize model, loss function and optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AttentionFusionModel(gene_dim=X_genomic.shape[1], chem_dim=X_chem.shape[1], dropout=0.2).to(device)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.0002, weight_decay=1e-5) # weight decay for regularization, no overfitting
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2) # learning rate scheduler to reduce LR if validation loss plateaus

epochs = 20

print("Start Training")

for epoch in range(epochs):
    # training loop
    model.train()
    train_loss = 0.0
    for batch_genes, batch_chem, batch_y in train_loader:
        batch_genes, batch_chem, batch_y = batch_genes.to(device), batch_chem.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        predictions = model(batch_genes, batch_chem)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * batch_genes.size(0)
    
    total_train_loss = train_loss / len(idx_train)
    
    # evaluation loop
    model.eval()
    test_loss = 0.0
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for batch_genes, batch_chem, batch_y in test_loader:
            batch_genes, batch_chem, batch_y = batch_genes.to(device), batch_chem.to(device), batch_y.to(device)
            predictions = model(batch_genes, batch_chem)
            
            all_preds.extend(predictions.cpu().numpy())
            all_targets.extend(batch_y.cpu().numpy())

    # convert lists to numpy arrays for metric calculations
    all_preds = np.array(all_preds)
    all_targets = np.array(all_targets)

    # Metrices
    test_rmse = root_mean_squared_error(all_targets, all_preds)
    test_r2 = r2_score(all_targets, all_preds)
    correlation = np.corrcoef(all_preds, all_targets)[0, 1] # pearson correlation as a simple measure of predictive performance in regression
    scheduler.step(test_rmse)
    print(f"Epoch {epoch+1:02d}/{epochs} | Train MSE: {total_train_loss:.4f} | Test RMSE: {test_rmse:.4f} | Test R²: {test_r2:.4f} | Test Corr: {correlation:.4f}")

Start Training
Epoch 01/20 | Train MSE: 2.5771 | Test RMSE: 1.7902 | Test R²: 0.5992 | Test Corr: 0.8766
Epoch 02/20 | Train MSE: 2.0191 | Test RMSE: 1.4970 | Test R²: 0.7197 | Test Corr: 0.8871
Epoch 03/20 | Train MSE: 1.9204 | Test RMSE: 1.4845 | Test R²: 0.7244 | Test Corr: 0.8930
Epoch 04/20 | Train MSE: 1.8624 | Test RMSE: 1.5043 | Test R²: 0.7170 | Test Corr: 0.8963
Epoch 05/20 | Train MSE: 1.7677 | Test RMSE: 1.4347 | Test R²: 0.7426 | Test Corr: 0.8997
Epoch 06/20 | Train MSE: 1.7036 | Test RMSE: 1.4246 | Test R²: 0.7462 | Test Corr: 0.8989
Epoch 07/20 | Train MSE: 1.6730 | Test RMSE: 1.6782 | Test R²: 0.6478 | Test Corr: 0.9042
Epoch 08/20 | Train MSE: 1.6331 | Test RMSE: 1.5798 | Test R²: 0.6879 | Test Corr: 0.9041
Epoch 09/20 | Train MSE: 1.6000 | Test RMSE: 1.6031 | Test R²: 0.6786 | Test Corr: 0.9076
Epoch 10/20 | Train MSE: 1.5002 | Test RMSE: 1.4155 | Test R²: 0.7494 | Test Corr: 0.9109
Epoch 11/20 | Train MSE: 1.4721 | Test RMSE: 1.5279 | Test R²: 0.7081 | Test Corr: 0.